# Multimodal Recommender Systems — Exercises

Companion to the note [Multimodal Recommender Systems](Multimodal%20Recommender%20Systems.md).

Practise the four-part multimodal pipeline (encoders → fusion → enhancement → optimisation): VBPR scoring and its parameter economy, early/late/attention fusion, LATTICE/FREEDOM-style item–item kNN graphs, contrastive (InfoNCE) modality alignment, and the question every multimodal paper must answer: do the modalities actually help? Item 'images' and 'texts' are synthetic pre-extracted feature vectors (stand-ins for frozen CLIP/BERT embeddings).

**17 exercises** · 🧠 Concept ×5 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import expit, softmax, log_softmax
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
rng = np.random.default_rng(0)

# Synthetic catalogue: 120 items with an 8-d 'image' embedding and a 6-d 'text' embedding.
# Users' taste depends on the image features (+ a small ID-only latent part); text is uninformative.
_g = np.random.default_rng(0)
NU, NI = 80, 120
f_img = _g.normal(size=(NI, 8))
f_txt = _g.normal(size=(NI, 6))
_aff = (_g.normal(size=(NU, 3)) @ _g.normal(size=(3, 8)) @ f_img.T / np.sqrt(8)
        + 0.5 * _g.normal(size=(NU, 2)) @ _g.normal(size=(NI, 2)).T + 0.3 * _g.normal(size=(NU, NI)))
X_all = (_aff > np.quantile(_aff, 0.85, axis=1, keepdims=True)).astype(float)   # implicit feedback
cold = np.arange(100, 120); warm = np.arange(100)        # items 100..119 are new: no training interactions
X_tr = X_all.copy(); X_tr[:, cold] = 0

def cold_auc(S):
    """Mean over users of P(score of a cold positive > score of a cold negative)."""
    a = []
    for u in range(NU):
        p, n = cold[X_all[u, cold] > 0], cold[X_all[u, cold] == 0]
        if len(p) and len(n):
            a.append(np.mean(S[u, p][:, None] > S[u, n][None, :]))
    return float(np.mean(a))

## Part A · Concepts

### Exercise 1 · Why (and when) multimodal?
*🧠 Concept · ★☆☆*

The note gives three reasons to add item content: cold-start items, sparse data, visually-driven taste.
For each, explain the mechanism in one sentence. Then name a domain where you would expect **little** gain from images, and why.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

ID embeddings are learned only from interactions. What does an item with 0 or 3 interactions get?

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **Cold-start items:** a new item has no learned ID embedding, but its image/text embedding exists on day 1, so the content term can score it immediately.
- **Sparse data:** items with few interactions have noisy ID embeddings. Content features act as a prior and share statistical strength between similar-looking items.
- **Visual taste:** in fashion/art/food, *what it looks like* is the reason people click, and pixels carry information no ID co-occurrence pattern has seen yet.

Little gain: e.g. **utility products** (phone chargers, batteries) or domains where taste is driven by price, brand or function, and also mature catalogues where every item already has
thousands of interactions. Here ID embeddings already capture everything the content would, which is the "tuned ID baselines close the gap" question in the note.

</details>

### Exercise 2 · Choosing a fusion strategy
*🧠 Concept · ★☆☆*

Which fusion strategy (early, late, attention, graph-based) matches each description, and what is its main risk?

(a) Per-modality recommenders each produce a score, combined by a weighted sum.
(b) Image and text vectors are concatenated and fed to one MLP.
(c) A per-user weight vector decides how much image vs text matters.
(d) An item–item kNN graph is built from each modality and used for convolution.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Look at the note's fusion table: 'one modality can dominate', 'misses cross-modal interactions', 'more parameters'.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Late fusion**: flexible and modular, but it misses cross-modal interactions (e.g. "red" in text + dress shape in image).
(b) **Early fusion**: simple, but a high-dimensional or large-scale modality can **dominate** the concatenation unless it is normalised (see Exercise 12).
(c) **Attention fusion**: personalised, but more parameters and it can overfit for sparse users.
(d) **Graph-based** (LATTICE/FREEDOM): state of the art on benchmarks, but extra graph construction cost and the $k$ / modality-weight hyperparameters.

</details>

### Exercise 3 · Why are encoders frozen?
*🧠 Concept · ★★☆*

In most multimodal recommenders (VBPR, MMGCN, LATTICE, FREEDOM…) the CNN/CLIP/BERT encoders are **frozen** and features are pre-extracted.
Give three reasons. What do you lose, and what is a middle ground?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about compute per training step, how many interactions per item there are, and catastrophic forgetting.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Compute:** running a ViT/BERT for every item in every BPR step is orders of magnitude slower than an embedding lookup. Pre-extracting once is cheap.
2. **Data:** interaction data is sparse. Fine-tuning a 100M-parameter encoder on BPR signals would overfit and destroy the general features (catastrophic forgetting).
3. **Reuse and reproducibility:** one feature file serves many models, and benchmarks (Amazon, TikTok) ship pre-extracted features.

**Loss:** the features are not adapted to the recommendation task (CLIP knows "dress", not "dresses this user buys").
**Middle ground:** keep the encoder frozen but learn a small projection on top (VBPR's $E$, an MLP adapter), or fine-tune only the last layers / LoRA adapters end to end.

</details>

### Exercise 4 · Designing an ablation
*🧠 Concept · ★★☆*

The note asks: "do multimodal features *actually* help, or do tuned ID-based baselines close the gap?"
Design a fair ablation study for a multimodal model with image + text + ID. Which variants do you train, how do you tune them, which metrics and splits do you report,
and which user/item subgroups would you break results down by?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Every variant deserves the same tuning budget. Think also about warm vs cold items.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **Variants:** ID-only (e.g. LightGCN / tuned MF), ID+image, ID+text, ID+image+text, and content-only (no ID) if relevant. Optionally, random features of the same
  dimension: if they "help" too, the gain comes from extra parameters, not information.
- **Tuning:** the same hyperparameter search budget for every variant, including the ID baseline (the most common flaw in papers).
- **Evaluation:** Recall@K / NDCG@K under **full ranking** with a temporal split ([[Evaluating Recommenders]]), several seeds, and confidence intervals.
- **Breakdowns:** cold vs warm items, items by popularity bucket, users by history length. Multimodal gains are typically concentrated in cold/tail items and vanish on the head.

</details>

### Exercise 5 · LATTICE vs FREEDOM
*🧠 Concept · ★★☆*

Both LATTICE (2021) and FREEDOM (2023) use item–item graphs built from modality similarity. What does LATTICE **learn** that FREEDOM **freezes**, and what does
FREEDOM **denoise** instead? Why can freezing make the model both faster and more accurate?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

LATTICE recomputes graph structure from learned modality projections during training.

</details>

<details>
<summary><b>✅ Solution</b></summary>

**LATTICE** learns the item–item graph: modality features pass through trainable projections, the kNN graph is rebuilt from the projected features during training, and
it is combined with the initial (raw-feature) graph.
**FREEDOM** observes that the learned graph adds little. It **freezes** the kNN graph built once from raw features, and instead **denoises the user–item graph** by
degree-sensitive edge pruning (dropping likely-noisy interactions during training).

Freezing removes the repeated $O(I^2)$ similarity computation per epoch (faster, less memory). It also removes a source of instability, since a graph built from
features that are still training is noisy early on. Pruning noisy user–item edges then targets a bigger error source than the item graph.

</details>

## Part B · By hand

### Exercise 6 · A VBPR score
*✍️ By hand · ★☆☆*

VBPR scores $\hat x_{ui}=\alpha+\beta_u+\beta_i+\gamma_u^\top\gamma_i+\theta_u^\top(E f_i)$. Compute $\hat x_{ui}$ for
$\alpha=0.5$, $\beta_u=0.2$, $\beta_i=-0.1$, $\gamma_u=(1, 0.5)$, $\gamma_i=(0.4, -0.2)$, $\theta_u=(1,-1)$,
$E=\begin{pmatrix}0.5&0&0.1\\0&1&0.2\end{pmatrix}$, $f_i=(1,0,2)$.

In [ ]:
x_vbpr = None

_E = np.array([[.5, 0, .1], [0, 1, .2]])
check('VBPR score', x_vbpr, .5 + .2 - .1 + np.array([1, .5]) @ np.array([.4, -.2]) + np.array([1, -1]) @ (_E @ np.array([1, 0, 2.])))

<details>
<summary><b>💡 Hint</b></summary>

$E f_i = (0.5+0.2,\ 0+0.4) = (0.7, 0.4)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\gamma_u^\top\gamma_i = 0.4-0.1 = 0.3$; $Ef_i = (0.7, 0.4)$, $\theta_u^\top Ef_i = 0.7-0.4 = 0.3$.
$\hat x_{ui} = 0.5+0.2-0.1+0.3+0.3 = 1.2$. In BPR training $\alpha$ and $\beta_u$ cancel in $\hat x_{ui}-\hat x_{uj}$, so they are usually dropped.

```python
x_vbpr = 1.2
```

</details>

### Exercise 7 · Why project the visual features?
*✍️ By hand · ★★☆*

$|U| = 10^5$ users, CNN features of dimension $F=4096$, visual factor dimension $D=64$.

1. Count the parameters of VBPR's visual part: $E\in\mathbb R^{D\times F}$ plus $\theta_u\in\mathbb R^D$ for every user.
2. Count the parameters if instead each user had a full preference vector $\theta_u\in\mathbb R^F$ over raw CNN features.
3. What is the ratio?

In [ ]:
params_vbpr = None
params_full = None
ratio = None

check('VBPR visual params', params_vbpr, 64 * 4096 + 1e5 * 64)
check('full params', params_full, 1e5 * 4096)
check('ratio', ratio, 1e5 * 4096 / (64 * 4096 + 1e5 * 64), tol=1e-2)

<details>
<summary><b>💡 Hint</b></summary>

$E$ is shared by all users; only $\theta_u$ scales with the number of users.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $64\cdot4096 + 10^5\cdot 64 = 262\,144 + 6\,400\,000 = 6\,662\,144$.
2. $10^5\cdot4096 = 409\,600\,000$.
3. Ratio $\approx 61.5$.
The shared projection $E$ learns *which visual directions matter for taste* once, from all users together, and each user only learns $D$ numbers. This is a low-rank factorisation of the
user × feature preference matrix, so it is cheaper and much less prone to overfitting for sparse users.

```python
params_vbpr = 6662144
params_full = 409600000
ratio = 409600000 / 6662144
```

</details>

### Exercise 8 · Ranking cold items with VBPR
*✍️ By hand · ★☆☆*

Two brand-new items have **no** interactions, so their $\beta_i$ and $\gamma_i$ are 0. Using $\theta_u$ and $E$ from Exercise 6, compute the visual scores
$\theta_u^\top E f$ for $f_1=(1,0,2)$ and $f_2=(0,1,0)$. Which one is ranked first? What would plain BPR-MF do?

In [ ]:
cold_scores = None   # [s1, s2]

_E = np.array([[.5, 0, .1], [0, 1, .2]]); _t = np.array([1, -1.])
check('visual scores', cold_scores, [_t @ _E @ np.array([1, 0, 2.]), _t @ _E @ np.array([0, 1, 0.])])

<details>
<summary><b>💡 Hint</b></summary>

$Ef_2 = (0, 1)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$s_1 = 0.3$, $s_2 = \theta_u^\top(0,1) = -1$, so item 1 is ranked first. Plain BPR-MF has no content term: both items score the same (0), so it cannot rank them at all.
That is the cold-start benefit of the content term.

```python
cold_scores = [0.3, -1.0]
```

</details>

### Exercise 9 · Late fusion vs attention fusion
*✍️ By hand · ★★☆*

For a user and an item, per-modality scores are image $0.8$, text $0.3$, ID $0.6$.

1. **Late fusion** with fixed weights $(0.5, 0.2, 0.3)$: compute the fused score.
2. **Attention fusion**: the user's attention logits are $(1, 0, 0.5)$; weights $=\mathrm{softmax}$(logits). Compute the weights and the fused score.

In [ ]:
late_score = None
att_weights = None   # 3 numbers
att_score = None

_s = np.array([.8, .3, .6]); _w = softmax([1, 0, .5])
check('late fusion', late_score, _s @ [.5, .2, .3])
check('attention weights', att_weights, _w, tol=1e-3)
check('attention score', att_score, _s @ _w, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$e^1 = 2.718$, $e^0 = 1$, $e^{0.5} = 1.649$; their sum is $5.367$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $0.5\cdot0.8 + 0.2\cdot0.3 + 0.3\cdot0.6 = 0.4+0.06+0.18 = 0.64$.
2. Weights $= (2.718, 1, 1.649)/5.367 = (0.506,\ 0.186,\ 0.307)$; score $= 0.8\cdot0.506+0.3\cdot0.186+0.6\cdot0.307 \approx 0.645$.
With attention, the weights are computed per user (or per user–item pair) from learned parameters, so a "visual" shopper and a "reads-the-description" shopper get different mixes.

```python
late_score = 0.64
att_weights = list(softmax([1, 0, 0.5]))
att_score = float(np.array([.8, .3, .6]) @ softmax([1, 0, 0.5]))
```

</details>

### Exercise 10 · A kNN item–item graph by hand
*✍️ By hand · ★★☆*

Four items have 2-d image features $a=(1,0)$, $b=(0.8,0.6)$, $c=(0,1)$, $d=(-1,0)$.
Compute all pairwise cosine similarities and, for each item, its **top-1 neighbour** (excluding itself). This is the LATTICE/FREEDOM graph with $k=1$.
Store the neighbour indices (a=0, b=1, c=2, d=3) in `top1`.

In [ ]:
top1 = None   # [nbr of a, nbr of b, nbr of c, nbr of d]

_F = np.array([[1, 0], [.8, .6], [0, 1], [-1, 0.]])
_nn = NearestNeighbors(n_neighbors=2, metric='cosine').fit(_F).kneighbors(_F, return_distance=False)
check('top-1 neighbours', top1, [r[1] if r[0] == k else r[0] for k, r in enumerate(_nn)])

<details>
<summary><b>💡 Hint</b></summary>

All vectors already have unit length, so cosine = dot product: $ab=0.8$, $ac=0$, $ad=-1$, $bc=0.6$, $bd=-0.8$, $cd=0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Similarities: $ab=0.8$, $ac=0$, $ad=-1$, $bc=0.6$, $bd=-0.8$, $cd=0$.
Top-1: $a\to b$ (0.8), $b\to a$ (0.8), $c\to b$ (0.6), $d\to c$ (0, best of $\{-1,-0.8,0\}$). So `top1 = [1, 0, 1, 2]`.
Note that $d$ gets a neighbour even though nothing is similar to it. kNN graphs always have exactly $k$ out-edges, which is one reason for thresholds or denoising.

```python
top1 = [1, 0, 1, 2]
```

</details>

### Exercise 11 · InfoNCE alignment by hand
*✍️ By hand · ★★☆*

Two items have L2-normalised image embeddings $v_1, v_2$ and text embeddings $t_1, t_2$ with cosine similarities
$\begin{pmatrix}v_1^\top t_1 & v_1^\top t_2\\ v_2^\top t_1 & v_2^\top t_2\end{pmatrix} = \begin{pmatrix}0.9&0.2\\0.1&0.8\end{pmatrix}$.
Compute the image→text InfoNCE loss $\frac12\sum_k -\log\frac{e^{v_k^\top t_k/\tau}}{\sum_j e^{v_k^\top t_j/\tau}}$ with $\tau = 0.5$.

In [ ]:
infonce_val = None

_S = np.array([[.9, .2], [.1, .8]]) / .5
check('InfoNCE', infonce_val, -np.mean(np.diag(log_softmax(_S, axis=1))), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

For two candidates, $-\log\frac{e^a}{e^a+e^b} = \log(1+e^{-(a-b)})$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Row 1: $a-b = (0.9-0.2)/0.5 = 1.4$, so the loss is $\log(1+e^{-1.4}) \approx 0.2204$. Row 2: $(0.8-0.1)/0.5 = 1.4$, the same, $0.2204$. Mean $\approx 0.220$.
Minimising it pulls each item's image and text embedding together and pushes them away from other items' embeddings. This is the "feature enhancement" stage
(SLMRec, BM3 without negatives, CLIP) ([[Self-Supervised and Contrastive Learning]]).

```python
infonce_val = np.log1p(np.exp(-1.4))
```

</details>

## Part C · Code from scratch

### Exercise 12 · Early fusion and modality dominance
*💻 Code · ★☆☆*

Implement `early_fusion(mods)` that **z-scores each modality** (per feature, statistics over all items), divides modality $m$ by $\sqrt{d_m}$ (so every modality has
the same expected squared norm), and concatenates. Then find the 5 most similar items (cosine, excluding itself) to cold item 100 in the fused space.

Afterwards try concatenating raw features where the text block is multiplied by 10 (a modality on a larger scale): how many of the 5 neighbours change?

In [ ]:
def early_fusion(mods):
    # mods: list of (n_items, d_m) arrays
    return None

fused = early_fusion([f_img, f_txt])
nbrs_100 = None   # 5 item indices

_Z = np.hstack([StandardScaler().fit_transform(m) / np.sqrt(m.shape[1]) for m in (f_img, f_txt)])
check('fused features', fused, _Z)
_s = cosine_similarity(_Z[100:101], _Z)[0]; _s[100] = -np.inf
check('5 nearest items to item 100', None if nbrs_100 is None else sorted(nbrs_100), sorted(np.argsort(-_s)[:5]))

<details>
<summary><b>💡 Hint</b></summary>

`(m - m.mean(0)) / m.std(0)` (population std, like `StandardScaler`), then `/ np.sqrt(m.shape[1])`, then `np.hstack`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Without normalisation, the cosine of the concatenation is dominated by the block with the largest norm. With the text block scaled ×10, 4 of the 5 neighbours change:
they are now determined mostly by text, which here is pure noise. Per-modality normalisation (or learned per-modality projections) is essential for early fusion.

```python
def early_fusion(mods):
    return np.hstack([(m - m.mean(0)) / m.std(0) / np.sqrt(m.shape[1]) for m in mods])

fused = early_fusion([f_img, f_txt])
_s = cosine_similarity(fused[100:101], fused)[0]; _s[100] = -np.inf
nbrs_100 = [int(i) for i in np.argsort(-_s)[:5]]

_raw = np.hstack([f_img, 10 * f_txt])
_s2 = cosine_similarity(_raw[100:101], _raw)[0]; _s2[100] = -np.inf
print('normalised:', nbrs_100, '| text x10:', np.argsort(-_s2)[:5].tolist(),
      '| changed:', 5 - len(set(nbrs_100) & set(np.argsort(-_s2)[:5].tolist())))
```

</details>

### Exercise 13 · A LATTICE-style multimodal item graph
*💻 Code · ★★☆*

Implement `knn_graph(F, k)`: cosine similarities between items, keep each item's top-$k$ neighbours (excluding itself) as 1s, **symmetrise**
($A\leftarrow\max(A, A^\top)$) and normalise $\tilde A = D^{-1/2}AD^{-1/2}$. Then build the multimodal graph
$\tilde A = w_{img}\tilde A_{img} + w_{txt}\tilde A_{txt}$ with $k=10$, $w=(0.7, 0.3)$.

Finally use it for **one propagation step** of item content: what fraction of item 100's top-10 neighbours in the image graph are also neighbours in the text graph?

In [ ]:
def knn_graph(F, k):
    return None

A_mm = None
overlap_100 = None

def _ref_knn(F, k):
    idx = NearestNeighbors(n_neighbors=k + 1, metric='cosine').fit(F).kneighbors(F, return_distance=False)
    A = np.zeros((len(F), len(F)))
    for r, row in enumerate(idx):
        A[r, [c for c in row if c != r][:k]] = 1
    A = np.maximum(A, A.T); d = A.sum(1)
    return A / np.sqrt(np.outer(d, d))
_Ai, _At = _ref_knn(f_img, 10), _ref_knn(f_txt, 10)
check('image kNN graph', knn_graph(f_img, 10), _Ai)
check('multimodal graph', A_mm, 0.7 * _Ai + 0.3 * _At)
check('overlap', overlap_100, len(set(np.flatnonzero(_Ai[100])) & set(np.flatnonzero(_At[100]))) / len(np.flatnonzero(_Ai[100])))

<details>
<summary><b>💡 Hint 1</b></summary>

`S = cosine_similarity(F)`; `np.fill_diagonal(S, -np.inf)`; `idx = np.argsort(-S, axis=1)[:, :k]`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`A[np.repeat(np.arange(n), k), idx.ravel()] = 1`, then `A = np.maximum(A, A.T)` and `d = A.sum(1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The image and text graphs barely overlap here (≈7% of item 100's image neighbours are also text neighbours), as expected since the synthetic text is unrelated to the image. On real data, overlap between modality graphs is a quick diagnostic:
very low overlap means the modalities carry different (or noisy) information, and the fusion weights matter a lot. LATTICE/FREEDOM then run LightGCN-style convolutions
$H \leftarrow \tilde A H$ on item embeddings over this graph ([[Deep Learning Recommenders]]).

```python
def knn_graph(F, k):
    S = cosine_similarity(F)
    np.fill_diagonal(S, -np.inf)
    idx = np.argsort(-S, axis=1, kind='stable')[:, :k]
    n = len(F)
    A = np.zeros((n, n))
    A[np.repeat(np.arange(n), k), idx.ravel()] = 1
    A = np.maximum(A, A.T)
    d = A.sum(1)
    return A / np.sqrt(np.outer(d, d))

A_img, A_txt = knn_graph(f_img, 10), knn_graph(f_txt, 10)
A_mm = 0.7 * A_img + 0.3 * A_txt
_ni, _nt = set(np.flatnonzero(A_img[100])), set(np.flatnonzero(A_txt[100]))
overlap_100 = len(_ni & _nt) / len(_ni)
print('overlap of image and text neighbourhoods of item 100:', round(overlap_100, 3))
```

</details>

### Exercise 14 · Attention fusion over modalities
*💻 Code · ★★☆*

Each user has a learned logit per modality (`att_logits`, shape users × 3 for image/text/ID), and per-modality score matrices `S_img, S_txt, S_id`
(users × items). Implement `attention_fuse(att_logits, [S_img, S_txt, S_id])` returning
$\hat s_{ui} = \sum_m \mathrm{softmax}(a_u)_m\,S^{(m)}_{ui}$, fully vectorised (no Python loop over users).

In [ ]:
_g = np.random.default_rng(6)
att_logits = _g.normal(size=(NU, 3))
S_img, S_txt, S_id = (_g.normal(size=(NU, NI)) for _ in range(3))

def attention_fuse(logits, score_mats):
    return None

S_fused = attention_fuse(att_logits, [S_img, S_txt, S_id])

_ref = np.array([softmax(att_logits[u]) @ np.stack([S_img[u], S_txt[u], S_id[u]]) for u in range(NU)])
check('fused scores', S_fused, _ref)

<details>
<summary><b>💡 Hint</b></summary>

`W = softmax(logits, axis=1)` (users × 3); stack scores to shape (3, users, items) and use `np.einsum('um,mui->ui', W, stack)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

`einsum` expresses "weight each modality's score matrix by the user's weight and sum" without loops. In real models the logits come from a small network of
the user (and item) embeddings and are trained end to end with the BPR loss.

```python
_g = np.random.default_rng(6)
att_logits = _g.normal(size=(NU, 3))
S_img, S_txt, S_id = (_g.normal(size=(NU, NI)) for _ in range(3))

def attention_fuse(logits, score_mats):
    W = softmax(logits, axis=1)
    return np.einsum('um,mui->ui', W, np.stack(score_mats))

S_fused = attention_fuse(att_logits, [S_img, S_txt, S_id])
```

</details>

### Exercise 15 · Symmetric InfoNCE (CLIP loss)
*💻 Code · ★★☆*

Implement `clip_loss(V, T, tau)` for batches of image embeddings `V` and text embeddings `T` ($B\times d$): L2-normalise rows, compute $S=VT^\top/\tau$,
and return the average of the image→text and text→image cross-entropies with the diagonal as targets.
Check that perfectly aligned pairs (`T = V`) give a lower loss than random pairs.

In [ ]:
def clip_loss(V, T, tau=0.1):
    return None

_g = np.random.default_rng(7)
V_b = _g.normal(size=(8, 16))
loss_aligned = clip_loss(V_b, V_b)
loss_random = clip_loss(V_b, _g.normal(size=(8, 16)))

def _ref(V, T, tau=0.1):
    V = V / np.linalg.norm(V, axis=1, keepdims=True); T = T / np.linalg.norm(T, axis=1, keepdims=True)
    S = V @ T.T / tau
    return 0.5 * (-np.mean(np.diag(log_softmax(S, axis=1))) - np.mean(np.diag(log_softmax(S, axis=0))))
check('aligned loss', loss_aligned, _ref(V_b, V_b))
if loss_aligned is not None:
    check('aligned < random', loss_aligned < loss_random, True)

<details>
<summary><b>💡 Hint</b></summary>

Text→image cross-entropy is a softmax over the **columns**: `log_softmax(S, axis=0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Aligned pairs give a loss close to 0 (each image's own caption wins the softmax by a large margin at $\tau=0.1$), while random pairs give ≈4.4. That is even above the chance level $\log B\approx2.08$, because with a small $\tau$ confidently wrong matches are punished hard.
The symmetric form treats both modalities as queries. Lower $\tau$ sharpens the softmax (see the temperature discussion in [[Deep Learning Recommenders]]).

```python
def clip_loss(V, T, tau=0.1):
    V = V / np.linalg.norm(V, axis=1, keepdims=True)
    T = T / np.linalg.norm(T, axis=1, keepdims=True)
    S = V @ T.T / tau
    i2t = -np.mean(np.diag(log_softmax(S, axis=1)))
    t2i = -np.mean(np.diag(log_softmax(S, axis=0)))
    return 0.5 * (i2t + t2i)

_g = np.random.default_rng(7)
V_b = _g.normal(size=(8, 16))
loss_aligned = clip_loss(V_b, V_b)
loss_random = clip_loss(V_b, _g.normal(size=(8, 16)))
print(round(loss_aligned, 4), round(loss_random, 4))
```

</details>

### Exercise 16 · VBPR from scratch: cold-start items
*💻 Code · ★★★*

Train VBPR with BPR-SGD on `X_tr` (cold items 100–119 have no interactions and are **not** sampled as negatives; sample $j$ from `warm`) using image features `f_img`:
$$\hat x_{ui} = \beta_i + \gamma_u^\top\gamma_i + \theta_u^\top E f_i$$
($K=4$ ID factors, $D=4$ visual factors, $\eta=0.05$, $\lambda=0.01$, 40 000 steps, init $\mathcal N(0,0.1^2)$, $\beta=0$).
Score the cold items with $\beta_i=0$, $\gamma_i=0$ (never trained), i.e. only the visual term, and report `cold_auc` (setup) as `auc_vbpr_cold`.

In [ ]:
def train_vbpr(X, f, K=4, D=4, lr=0.05, lam=0.01, steps=40000, seed=0):
    # TODO: return beta_i, gamma_u, gamma_i, theta_u, E
    return None

auc_vbpr_cold = None

_g = np.random.default_rng(0)
print('random scores, cold AUC:', round(cold_auc(_g.normal(size=(NU, NI))), 3))
if auc_vbpr_cold is not None:
    check('VBPR ranks unseen items well (cold AUC > 0.8)', auc_vbpr_cold > 0.8, True)
else:
    check('VBPR', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

With $\Delta f = f_i - f_j$: $\hat x_{uij} = \beta_i-\beta_j + \gamma_u^\top(\gamma_i-\gamma_j) + \theta_u^\top E\Delta f$ and $s=\sigma(-\hat x_{uij})$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Gradient steps (ascent on $\ln\sigma$): $\theta_u \mathrel{+}= \eta(s\,E\Delta f - \lambda\theta_u)$, $E \mathrel{+}= \eta(s\,\theta_u\Delta f^\top - \lambda E)$; the $\beta,\gamma$ updates are as in BPR-MF. Use the old $\gamma_u,\theta_u$ on the right-hand sides.

</details>

<details>
<summary><b>✅ Solution</b></summary>

VBPR ranks never-seen items with cold AUC ≈ 0.91, while an ID-only model gives every cold item the same score (AUC 0.5). The projection $E$ and the user vectors $\theta_u$ were learned on
warm items, and transfer to new ones because the taste lives in the visual space. In the multi-stage picture this is the content side of the hybrid for new items
([[Content-Based and Hybrid Recommenders]]).

```python
def train_vbpr(X, f, K=4, D=4, lr=0.05, lam=0.01, steps=40000, seed=0):
    g = np.random.default_rng(seed)
    nu, ni = X.shape
    gu = 0.1 * g.normal(size=(nu, K)); gi = 0.1 * g.normal(size=(ni, K))
    th = 0.1 * g.normal(size=(nu, D)); E = 0.1 * g.normal(size=(D, f.shape[1])); bi = np.zeros(ni)
    us, its = np.nonzero(X)
    for _ in range(steps):
        t = g.integers(len(us)); u, i = us[t], its[t]
        j = warm[g.integers(len(warm))]
        while X[u, j]:
            j = warm[g.integers(len(warm))]
        df = f[i] - f[j]
        x = bi[i] - bi[j] + gu[u] @ (gi[i] - gi[j]) + th[u] @ (E @ df)
        s = expit(-x)
        guu, thu = gu[u].copy(), th[u].copy()
        bi[i] += lr * (s - lam * bi[i]); bi[j] += lr * (-s - lam * bi[j])
        gu[u] += lr * (s * (gi[i] - gi[j]) - lam * gu[u])
        gi[i] += lr * (s * guu - lam * gi[i]); gi[j] += lr * (-s * guu - lam * gi[j])
        th[u] += lr * (s * (E @ df) - lam * th[u])
        E += lr * (s * np.outer(thu, df) - lam * E)
    return bi, gu, gi, th, E

bi_v, gu_v, gi_v, th_v, E_v = train_vbpr(X_tr, f_img)
bi_v[cold] = 0; gi_v[cold] = 0        # never-trained items: only the visual term remains
S_vbpr = bi_v[None, :] + gu_v @ gi_v.T + th_v @ (E_v @ f_img.T)
auc_vbpr_cold = cold_auc(S_vbpr)
print('VBPR cold AUC:', round(auc_vbpr_cold, 3))
```

</details>

### Exercise 17 · Modality ablation
*💻 Code · ★★★*

A cheap content-only recommender: z-score a modality's features (statistics over **warm** items), build each user's profile as the mean feature vector of their training positives,
and score items by the dot product with the profile. Implement `profile_scores(F)` and compute the cold AUC with image only (`auc_img`), text only (`auc_txt`),
and both concatenated (`auc_both`). What does the ablation tell you, and what is the general lesson for multimodal papers?

In [ ]:
def profile_scores(F):
    return None

auc_img = auc_txt = auc_both = None

if auc_img is not None:
    def _ps(F):
        Z = (F - F[warm].mean(0)) / F[warm].std(0)
        return (X_tr @ Z / np.maximum(X_tr.sum(1, keepdims=True), 1)) @ Z.T
    check('image-only AUC', auc_img, cold_auc(_ps(f_img)))
    check('text-only AUC', auc_txt, cold_auc(_ps(f_txt)))
    check('adding the useless modality hurts', auc_both < auc_img, True)
else:
    check('ablation', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`Z = (F - F[warm].mean(0)) / F[warm].std(0)`; `P = X_tr @ Z / X_tr.sum(1, keepdims=True)`; scores `P @ Z.T`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Image only ≈ 0.92, text only ≈ 0.43 (chance level, since the text is noise), both ≈ 0.89. **Adding a modality made results worse**: the noisy text dimensions dilute the
dot product. More modalities do not automatically help. Each must earn its place in an ablation with equal tuning, ideally broken down by cold and warm items.

```python
def profile_scores(F):
    Z = (F - F[warm].mean(0)) / F[warm].std(0)
    P = X_tr @ Z / np.maximum(X_tr.sum(1, keepdims=True), 1)
    return P @ Z.T

auc_img = cold_auc(profile_scores(f_img))
auc_txt = cold_auc(profile_scores(f_txt))
auc_both = cold_auc(profile_scores(np.c_[f_img, f_txt]))
print('image', round(auc_img, 3), '| text', round(auc_txt, 3), '| both', round(auc_both, 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Multimodal Recommender Systems](Multimodal%20Recommender%20Systems.md).*